# Bundled Exasol lesson: Translation model

Requires a configured Transformers Extension, model downloads and its language container. These services are not installed by opening a notebook.

Run cells from top to bottom. In Marimo each editable cell has its own Run button; opening a lesson executes no example code. Widget steps retain their Python code; use the matching Jupyter notebook for the original widget UI.

For dashboards use the kit Data Scientist 03_share flow on Dash Server.

Source: [Exasol Notebook Connector](https://github.com/exasol/notebook-connector/blob/14e2c7ad088b7109aa21d283e98f3126bbbe27e5/exasol/nb_connector/resources/notebooks/transformers/translation.ipynb). MIT license: LICENSE.txt.

In [ ]:
from exakit.nb.example_runtime import bootstrap
ai_lab_config = bootstrap(globals(), 'transformers/translation.ipynb')

<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

# Translation model

In this notebook we will load and use a language translation model. Learn more about the Translation task <a href="https://huggingface.co/tasks/translation" target="_blank" rel="noopener">here</a>. Please also refer to the Transformer Extension <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a> to find more information about the UDF used in this notebook.

We will be running SQL queries using <a href="https://github.com/ploomber/jupysql" target="_blank" rel="noopener">JupySQL</a>SQL Magic.

## Prerequisites

Prior to using this notebook the following steps need to be completed:
1. [Configure the AI Lab](../main_config.ipynb).
2. [Initialize the Transformer Extension](te_init.ipynb).

## Setup

### Open Secure Configuration Storage

In [ ]:
print("Using the Starter Kit encrypted connection; tutorial schema: EXAKIT_EXAMPLES.")

Let's bring up JupySQL and connect to the database via SQLAlchemy. 
Please refer to the documentation of [sqlalchemy-exasol](https://github.com/exasol/sqlalchemy-exasol) for details on how to connect to the database using the Exasol SQLAlchemy driver.

In [ ]:
from exakit.nb.example_runtime import init_sql as jupysql_init
jupysql_init(ai_lab_config)

# AI_TRANSLATE

There are two UDfs for translation available. The `AI_TRANSLATE` udf uses some default configuration parameters and a preselected model. If you need more control over your prediction, the `AI_TRANSLATE_EXTENDED` udf allows you to set all configuration parameters.

We will start by showing you the `AI_TRANSLATE` UDF.

This udf uses the translation pipeline(previously provided by transformers. we now maintain it ourselves) with the [google-t5/t5-base](https://huggingface.co/google-t5/t5-base) model for the prediction.
                                                                                        We will now use the `AI_TRANSLATE` UDF to translate some text from English to German.
                                                                                          The udf takes various input parameters:

 * text_data: The text to translate.
 * source_language: The language of the input.
 * target_language: The language of the desired output.


Source and target language should be known by the model, check the models documentation if you are unsure.                                                                    

In [ ]:
# Here is the text in English
MY_TEXT = 'We all live in a yellow submarine'

# Make sure our text can be used in an SQL statement.
MY_TEXT = MY_TEXT.replace("'", "''")

SOURCE_LANGUAGE = 'English'
TARGET_LANGUAGE = 'German'

You need to supply these parameters in the correct order. Further information can be found in the  <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.

Let's run the translation model.
We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_TRANSLATE(
        '{{MY_TEXT}}',
        '{{SOURCE_LANGUAGE}}',
        '{{TARGET_LANGUAGE}}'
    )
)
SELECT translation_text, error_message FROM MODEL_OUTPUT

The translated text can be found in the output column `TRANSLATION_TEXT`, and any error occurring while executing the udf will be saved in the `ERROR_MESSAGES` column.

We select only some of the udf's output columns in these examples.  If you need more details, you can find information on all available output columns in the <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.

The code above shows how the model works on a toy example. However, the main purpose of having a model deployed in the database is to get a quick response for a batch input. The performance gain comes from two factors - localization and parallelization. The first means that the input data never crosses the machine boundaries. The second means that multiple instances of the model are processing the data on all available nodes in parallel.

Another advantage of making predictions within the database is the enhanced data security. The task of safeguarding privacy can be simplified giving the fact that the source data never leaves the database machine.

A more practical application would involve translating multiple pieces of text stored in a column of a database table. For example, we might want to do the translation for each row of the input table `MY_TEXT_TABLE`, where the text in the original language is stored in the column `MY_ORIGINAL_TEXT`. The SQL for this task would look similar to this:
```
SELECT AI_TRANSLATE(MY_ORIGINAL_TEXT, ...) FROM MY_TEXT_TABLE;
```
Please note, that the response time observed on the provided example with a single input will not be scaled up linearly in case of multiple inputs. Much of the latency falls on loading the model into the CPU memory from BucketFS. This needs to be done only once regardless of the number of inputs.

# AI_TRANSLATE_EXTENDED

If you want to use a different model for prediction, you can use the `AI_TRANSLATE_EXTENDED` udf instead. This will work nearly the same, but you need to have a model installed and then set the udf parameters so the model can be found by the udf.

## Get a language model

To demonstrate the translation task we will use [Facebook WMT19 English-German model](https://huggingface.co/facebook/wmt19-en-de).

We need to load the model from the Hugging Face Hub into the [BucketFS](https://docs.exasol.com/db/latest/database_concepts/bucketfs/bucketfs.htm). This could potentially be a long process, depending on the speed of the database connection. Unfortunately, we cannot tell exactly when it has finished. The notebook's hourglass may not be a reliable indicator. BucketFS will still be doing some work when the call issued by the notebook returns. Please wait for a few moments after that, before querying the model.

You might see a warning that some weights are newly initialized and the model should be trained on a down-stream task. Please ignore this warning. For the purpose of this demonstration, it is not important, the model should still be able to produce some meaningful output.

In [ ]:
from exasol.nb_connector.model_installation import install_model, TransformerModel
from transformers import AutoModelForSeq2SeqLM

# This is the name of the model at the Hugging Face Hub
MODEL_NAME = 'facebook/wmt19-en-de'
install_model(ai_lab_config, TransformerModel(MODEL_NAME, 'translation', AutoModelForSeq2SeqLM))

## Use the language model

We will now use the `AI_TRANSLATE_EXTENDED` UDF to translate the same text as before. In addition to the previous parameters, we also need to set `max_new_tokens`, which will control the maximal length of output the model can return per input text.

In [ ]:
MAX_NEW_TOKENS = 200

The udf takes various input parameters:

* device_id: To run the UDF on a GPU, specify the valid cuda device ID.
* bucketfs_conn: The BucketFS connection name.
* sub_dir: The directory where the model is stored in the BucketFS.
* model_name: The name of the model to use for prediction.
* text_data: The text to translate.
* source_language: The language of the input. Required for multilingual models only.
* target_language: The language of the desired output. Required for multilingual models only.
* max_new_tokens: The maximum amount of generated tokens.

You need to supply these parameters in the correct order. Further information can be found in the  <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.

Let's run the translation model.
We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_TRANSLATE_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        '{{MY_TEXT}}',
        '{{SOURCE_LANGUAGE}}',
        '{{TARGET_LANGUAGE}}',
        {{MAX_NEW_TOKENS}}
    )
)
SELECT translation_text, error_message FROM MODEL_OUTPUT